In [1]:
! pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 MB 5.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.9 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.5-py2.py3-none-any.whl size=317747860 sha256=a059d582f0a6663fb315ab7b04fd0e8740d8b635aefdbbfa77f4480c08188c79
  Stored in directory: /home/jovyan/.cache/pip/wheels/0c/7f/b4/0e68c6d8d89d2e582e5498ad88616c16d7c19028680e9d3840
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = SparkSession.builder \
    .appName("MySparkApp") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memoryOverhead", "8g") \
    .config("spark.driver.memoryOverhead", "8g") \
    .config("spark.cores.max", "16") \
    .config("spark.task.maxFailures", "5") \
    .config("spark.sql.shuffle.partitions", "500") \
    .config("spark.network.timeout", "600s") \
    .config("spark.default.parallelism", "1000") \
    .config("spark.driver.maxResultSize", "10g") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()


hdfs_path = "hdfs://namenode:8020/user/hadoop/csv_files"
skills_df = spark.read.csv(hdfs_path + "/job_skills.csv", header=True, inferSchema=True)
postings_df = spark.read.csv(hdfs_path + "/linkedin_job_postings.csv", header=True, inferSchema=True)

df = skills_df.join(postings_df, on="job_link", how="inner").filter((col("job_type").isNotNull()) & 
    (col("job_skills").isNotNull()) & 
    (col("job_title").isNotNull()))



In [6]:
data_df = df.limit(1000)

tokenizer = Tokenizer(inputCol="job_skills", outputCol="skills_tokens")
hashing_tf = HashingTF(inputCol="skills_tokens", outputCol="skills_tf", numFeatures=1000)
idf = IDF(inputCol="skills_tf", outputCol="features")
label_indexer = StringIndexer(inputCol="job_title", outputCol="label", handleInvalid="keep")


pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf, label_indexer])
data_prepared = pipeline.fit(data_df).transform(data_df)

data_prepared_subset = data_prepared

In [7]:
train_data, test_data = data_prepared_subset.randomSplit([0.8, 0.2], seed=42)


In [8]:


rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100)
model = rf.fit(train_data)

predictions = model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = evaluator.evaluate(predictions)

print(f"F1 Score: {f1}")


F1 Score: 0.517597199189239


In [9]:
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = accuracy_evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy}")


Accuracy: 0.654320987654321


In [1]:

paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [50, 100, 200])
             .addGrid(rf.maxDepth, [5, 10, 20])
             .build())

crossval = CrossValidator(estimator=rf,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

cvModel = crossval.fit(train_data)

cv_predictions = cvModel.transform(test_data)
best_f1 = evaluator.evaluate(cv_predictions)
print(f"Best F1 Score after hyperparameter tuning: {best_f1}")


Best F1 Score after hyperparameter tuning: 0.6632560987654355


In [23]:
cvModel.bestModel.save("hdfs://namenode:8020/user/hadoop/csv_files/Machine_Learning/models/job_title_prediction_rf")
cv_predictions.select("job_link", "prediction").write.mode("overwrite").csv(
    "hdfs://namenode:8020/user/hadoop/csv_files/Machine_Learning/outputs/job_title_predictions",
    header=True
)
